# Feature Selection

## Introduction to Core Data

![glomar](http://deepseadrilling.org/images/challenger1.jpg)<br>
One of the objectives of this module is to familiarize yourself with new data types. Geologist learn a lot about the history of the Earth by studying core material, i.e. cylindrical pieces of rock recovered from the subsurface by a drill string. Drilling for cores can be performed on land, or at sea from a drilling vessel. One famous drilling vessel is the R/V JOIDES Resolution. I also used to work as a staff scientist on the JOIDES Resolution (aka "JR") before joining Imperial College, so I know it well. The JR is operated by the <a href="https://www.iodp.org/">International Ocean Discovery Program (IODP, 2003-today)</a>, which was preceded by the <a href="http://www-odp.tamu.edu/">Ocean Drilling Program (ODP, 1985-2003)</a> and the <a href="http://deepseadrilling.org/about.htm">Deep Sea Drilling Project (DSDP, 1966-1985)</a>. The various version of the ocean drilling program thus represents one of the major investment in Earth Science over the last 55 years, and has acquired a treasure trove of core and logging data. We will take full advantage of this here.<br>

![cores](https://www.iodp.org/images/right_side_random_images/357_mircobio_carolcotterill.jpg)<br>

Because we will be using ODP/IODP data extensively in this course it is worth defining what the different terms represent.

'Leg' refers to the particular expedition the drill ship was on. In more recent IODP cruises, this is referred to as the 'Expedition' or 'Exp'. Think of a leg as a 6-9 week-long cruise with a single crew, addressing a single research objective. Leg or Expedition are represented by a sequential <code>Int</code>.

'Site' means the name of the general area that was drilled, which is also a sequential number. At each site, the vessel can drill multiple wells, known a 'Hole' and often represented with the letter 'H'. The wells are represented by an ordered <code>str</code>: 'A', 'B', 'C', etc..

Although a drilling vessel aims to recover a continuous section of rock, for practical reasons this is not possible. Thus, pipes are added to the drill string as coring down continues, and a length of rock nominally the length of the drilling pipe is recovered. The pipes on the JR are 9.8 meters, and so we recover cores of 9.8 meters. Cores are denoted with a unique number ('Cor'), followed by the tool used for coring ('T'), which is a 1 character designation of the drilling technique used (more about this later in the course). 'Sc' designates the section: each 9.8 meters-long core is cut into 1.5 meters sections for easy handling, with the addition of the core catcher 'CC' which represents material not in the core liner (in oder words, we are not sure where this material comes from - the core itself, the side of the hole, or the seafloor). For each sample from the ocean drilling program, a 'Top(cm)' is defined and represents the cm down from the top of the section where the sample is located.

Thus, an ODP/IODP sample can be fully spatially resolved by the combination of the parameters we have described above (and that are always supplied as part of the dataset as a unique identifyer). For instance, an example of a sample would be ODP Sample **198-1207-A-2-H-2-65**. This means that this particular sample comes from ODP Leg 198, Site 1207, Hole A (or 'well A'), core 2 drilling with a piston corer ('H'), section 2, 65 cm down section. With this information the 'Depth (mbsf)' which is the depths below seafloor of any sample can be calculated.

Multiple sets of properties can be measured on a core sample, either onboard the ship or postcruise, once the scientists are back onshore. All of this information can be precisely correlated and used by data scientists like yourselves.

![cores](https://iodp.tamu.edu/database/wholecore.jpg)

### 🎯 Identify useful predictors and compare full and reduced models.
We use the raw core measurements so missing responses are excluded and learned preprocessing can be fitted within each cross-validation fold. The supplied `clean_dataset.csv` was preprocessed in advance and is unsuitable for this comparison.

For more preprocessing practice, see `OptionalE04.ipynb`.

👇 Load the raw core data and inspect the first five rows.

In [ ]:
import pandas as pd

data = pd.read_csv('../../Excercise_data/L05/core_data.csv').drop_duplicates()
data = data.dropna(subset=['NGR total counts (cps)', 'Depth CSF-A (m)'])
data.head()

We want to predict the natural gamma ray response (`NGR total counts (cps)`) from depth, reflectance and coring type. Do not impute missing target values: an invented response is not an observed training label.

The setup below reserves a test set and a validation set. Explore and fit using `X_train`; use validation for feature inspection. Keep `X_test` untouched until your modelling decisions are fixed. This random split studies prediction within the sampled holes; generalisation to a new hole would require a grouped split.

In [ ]:
from sklearn.model_selection import train_test_split, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_features = ['Depth CSF-A (m)', 'Reflectance L*', 'Reflectance a*', 'Reflectance b*']
X = data[numeric_features + ['Type']].copy()
y = data['NGR total counts (cps)'].copy()
X_development, X_test, y_development, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_development, y_development, test_size=0.25, random_state=42)

numeric = Pipeline([('imputer', SimpleImputer(strategy='median')),
                    ('scaler', StandardScaler())])
categorical = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
preprocessor = ColumnTransformer([
    ('num', numeric, numeric_features), ('cat', categorical, ['Type'])])
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 🔍 Collinearity Investigation
1. Calculate the Pearson correlation matrix of `X_train[numeric_features]` and plot a heatmap. Use only training observations for this exploration.

<details>
    <summary>💡 Hint</summary>
ℹ️ The easiest way to draw heatmaps is using the Seaborn <code>heatmap()</code> methjod. <a href='https://seaborn.pydata.org/generated/seaborn.heatmap.html'>Read the doc</a> and do all of the necessary imports. Don't forget that you will need to also obtain a correlation matrix of your features.
</details>

In [ ]:
# your code

2. Visualize Pairwise Correlations
Use Seaborn’s pairplot() to explore pairwise relationships between the numerical training features, which will provide insight into feature distributions and potential collinear relationships.

<details>
    <summary>💡 Hint</summary>
ℹ️ You should investigate the Seaborn <a href='https://seaborn.pydata.org/generated/seaborn.pairplot.html'> <code>pairplot()</code></a> method.
</details>

In [ ]:
# your code

3. Identify Highly Correlated Features
- Identify pairs with absolute correlation above 0.75.
- Suggest which member you might remove and explain why.
- Keep the full feature set for the baseline below. A data-derived selection rule must be fitted inside each CV fold; dropping columns after inspecting all folds would bias the comparison.

In [ ]:
# your code

# 🏆 Base Model
- Build a `Pipeline` with the supplied `preprocessor` and `LinearRegression()`.
- Use `cross_val_score` on raw `X_train` and `y_train` with `cv=cv` and `scoring='r2'`. Each fold must refit preprocessing.
- Save the **mean** fold score as `base_model_score` and report the standard deviation. Do not select the best fold.

In [ ]:
# your code

# 🔍 Feature Permutation
- Fit the baseline pipeline on `X_train, y_train`.
- Compute `permutation_importance` on `X_val, y_val` with `scoring='r2'`, `n_repeats=10` and `random_state=42`.
- Rank the original input features by the mean reduction in R² and report variability. Save the highest-ranked feature name as `best_feature`.

This measures reliance of this fitted model, not intrinsic or causal importance. Correlated features can substitute for one another. Validation-guided selection requires an independent final test.

In [ ]:
# your code

# 🎯 Simplified Model with Important Features
- Build a second pipeline: `preprocessor` → `SelectKBest(f_regression, k=5)` → `LinearRegression()`. Import the selector and scoring function from `sklearn.feature_selection`.
- Run the same CV on raw `X_train, y_train`. The selector learns its ranking separately in each training fold.
- Save the **mean** R² as `simplified_model_score` and report its standard deviation.
- Compare with the baseline. Does fewer features help here? Does a univariate ranking necessarily capture every useful combination?

After fixing your choice, fit it on `X_development, y_development` and evaluate once on `X_test, y_test`.

In [ ]:
# your code

A simpler model may retain similar performance, but this is an empirical question. Discuss the size of the difference alongside fold-to-fold variation; CV standard deviation is not a confidence interval.

# Reducing Dimensionality for Classification
In this exercise, we’ll reduce dimensionality by removing correlated and unimportant features, then evaluate model performance on both the full and reduced feature sets.

## Setup Code:

Run the setup code to load and split the data. For a manageable SVM comparison, use a stratified 5,000-row training subset for the CV experiments, then refit the selected approach on the full training set before final testing:

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

df = pd.read_csv('../../Excercise_data/L05/wines_binary.csv')
y = df['is_good_quality']
X = df.drop(columns=['is_good_quality'])
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_train, _, y_train, _ = train_test_split(
    X_train_full, y_train_full, train_size=5000,
    random_state=42, stratify=y_train_full)
classification_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

## Feature Correlation and Model-Based Selection
- Explore correlations in `X_train` and discuss possible redundancy; keep all columns for the baseline.
- Evaluate a pipeline containing `StandardScaler()` and `LogisticRegression(max_iter=3000)` using `classification_cv`.
- For feature selection, insert `SelectFromModel(LogisticRegression(max_iter=3000))` after scaling and before the final classifier. Keep the selector **inside** the pipeline passed to CV.
- Compare full and selected pipelines with the same folds for logistic regression, a decision tree and an SVM. Set `random_state=42` where applicable.
- Compare mean accuracy and its fold variation; after choosing the approach, refit on `X_train_full, y_train_full` and report test performance once.

Coefficient magnitudes depend on feature units. Standardisation helps compare scales, but correlated predictors can still share or exchange coefficients; magnitude is not a causal importance measure.

In [ ]:
# your code

## Observations:
- Compare the accuracy of each model on both the full and reduced feature sets.
- Observe if dimensionality reduction (dropping less important features) affected the model’s accuracy positively, negatively, or made no significant difference.







In [ ]:
# your code